# 04 — Final visuals and tables

**Purpose:** regenerate polished, traceable submission figures. Method 1 is a focused comparative EDA: Australia is shown against the same-period supplied-country median and interquartile range for the four pre-specified outcomes. Social outcomes are independent three-year pooled windows, not annual observations.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'OECD Data.csv'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='talk')
from src.oecd_audit import load_clean

df = load_clean(RAW_FILE)

## Method 1 — Focused comparative EDA

The four panels retain native units; they are not indexed or combined into a score. The shaded band is the reference interquartile range (IQR), and reference composition can change when countries do not report an outcome in a period. The generated coverage table records the country count for every plotted point.

In [ ]:
PRIMARY_SPECS = {
    '1_1': ('Household income per person', 'USD, PPP', False),
    '2_1': ('Employment rate', 'Percent', False),
    '7_1_DEP': ('Lack of social support', 'Percent', True),
    '11_2': ('Negative affect', 'Percent', True),
}
WINDOW_MIDPOINTS = {
    '2008-10': 2009, '2011-13': 2012, '2014-16': 2015,
    '2017-19': 2018, '2020-22': 2021, '2023-25': 2024,
}
WINDOW_LABELS = {
    2009: '2008–10', 2012: '2011–13', 2015: '2014–16',
    2018: '2017–19', 2021: '2020–22', 2024: '2023–25',
}


def prepare_trajectory(data, code):
    """Return Australia and same-period reference summaries through 2024."""
    subset = data.loc[data['indicator_code'].eq(code) & data['year'].le(2024)].copy()
    if PRIMARY_SPECS[code][2]:
        subset = subset.sort_values('year').drop_duplicates(
            ['country_code', 'independent_period'], keep='last'
        )
        subset['plot_period'] = subset['independent_period'].map(WINDOW_MIDPOINTS)
    else:
        subset['plot_period'] = subset['year']
    australia = subset.loc[subset['country_code'].eq('AUS'), ['plot_period', 'value']]
    australia = australia.rename(columns={'value': 'australia_value'})
    reference = (
        subset.loc[subset['country_code'].ne('AUS')]
        .groupby('plot_period', as_index=False)
        .agg(
            reference_median=('value', 'median'),
            reference_q25=('value', lambda x: x.quantile(0.25)),
            reference_q75=('value', lambda x: x.quantile(0.75)),
            reference_country_count=('country_code', 'nunique'),
        )
    )
    result = australia.merge(reference, on='plot_period', how='inner')
    result.insert(0, 'indicator_code', code)
    result.insert(1, 'indicator', data.loc[data['indicator_code'].eq(code), 'indicator'].iloc[0])
    return result.sort_values('plot_period').reset_index(drop=True)


trajectories = pd.concat([prepare_trajectory(df, code) for code in PRIMARY_SPECS], ignore_index=True)

# Reliability checks: coverage, reference summaries and independent social windows.
assert set(trajectories['indicator_code']) == set(PRIMARY_SPECS)
assert trajectories['reference_country_count'].ge(1).all()
assert trajectories[['australia_value', 'reference_median', 'reference_q25', 'reference_q75']].notna().all().all()
assert (trajectories['reference_q25'] <= trajectories['reference_median']).all()
assert (trajectories['reference_median'] <= trajectories['reference_q75']).all()
expected_endpoints = {
    '1_1': (44625.0, 50629.0), '2_1': (75.444, 80.262),
    '7_1_DEP': (4.926543, 10.043193), '11_2': (12.244020, 14.853043),
}
for code, (first, last) in expected_endpoints.items():
    values = trajectories.loc[trajectories['indicator_code'].eq(code), 'australia_value']
    np.testing.assert_allclose([values.iloc[0], values.iloc[-1]], [first, last], rtol=0, atol=1e-5)
for code in ['7_1_DEP', '11_2']:
    periods = trajectories.loc[trajectories['indicator_code'].eq(code), 'plot_period'].tolist()
    assert periods == [2009, 2012, 2015, 2018, 2021, 2024]
for code in ['1_1', '2_1']:
    periods = trajectories.loc[trajectories['indicator_code'].eq(code), 'plot_period']
    assert periods.between(2010, 2024).all()

coverage = trajectories[['indicator_code', 'indicator', 'plot_period', 'reference_country_count']].copy()
coverage = coverage.rename(columns={'plot_period': 'period'})
coverage['period_label'] = np.where(
    coverage['indicator_code'].isin(['7_1_DEP', '11_2']),
    coverage['period'].astype(str) + ' pooled-window midpoint',
    coverage['period'].astype(str),
)
coverage_path = TABLE_DIR / 'material_social_trajectory_coverage.csv'
coverage.to_csv(coverage_path, index=False)

# Okabe–Ito colours improve accessibility for many colour-vision deficiencies.
AUSTRALIA_COLOUR, MEDIAN_COLOUR, IQR_COLOUR = '#D55E00', '#0072B2', '#56B4E9'
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, (code, (title, unit, pooled)) in zip(axes.flat, PRIMARY_SPECS.items()):
    plot_data = trajectories.loc[trajectories['indicator_code'].eq(code)]
    ax.fill_between(plot_data['plot_period'], plot_data['reference_q25'], plot_data['reference_q75'], color=IQR_COLOUR, alpha=0.28, label='Reference IQR')
    ax.plot(plot_data['plot_period'], plot_data['reference_median'], marker='o', color=MEDIAN_COLOUR, linewidth=2, label='Reference median')
    ax.plot(plot_data['plot_period'], plot_data['australia_value'], marker='o', color=AUSTRALIA_COLOUR, linewidth=2.8, label='Australia')
    # Show the coverage range once so annual panels stay readable.
    count_min = int(plot_data['reference_country_count'].min())
    count_max = int(plot_data['reference_country_count'].max())
    count_label = f'Reference n={count_min}' if count_min == count_max else f'Reference n={count_min}–{count_max}'
    ax.text(0.99, 0.03, count_label, transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color=MEDIAN_COLOUR)
    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_ylabel(unit)
    if pooled:
        periods = plot_data['plot_period'].tolist()
        ax.set_xticks(periods, [WINDOW_LABELS[year] for year in periods])
        ax.tick_params(axis='x', labelrotation=15, labelsize=9)
        ax.set_xlabel('Three-year pooled window')
    else:
        ax.set_xlabel('Year')
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].legend(frameon=False, fontsize=9, loc='best')
fig.suptitle('Australia versus same-period supplied-country reference distributions', fontsize=16, fontweight='bold')
fig.text(0.5, -0.015, 'Shaded band: supplied-country IQR. Reference n excludes Australia. Social points represent independent pooled windows plotted at their midpoints.', ha='center', fontsize=10)
figure_path = FIGURE_DIR / 'material_social_trajectories.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

display(coverage)
print(f'Wrote {figure_path.relative_to(PROJECT_ROOT)} and {coverage_path.relative_to(PROJECT_ROOT)}')